In [10]:
# ==================================================================================================
# CELL 1 — IMPORT REQUIRED LIBRARIES
# ==================================================================================================

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("=" * 100)
print("PHASE 03.04 — BASELINE CROSS-MODAL TRAINING")
print("=" * 100)

print("\nLIBRARY STATUS")
print("-" * 100)

print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"PyTorch     : {torch.__version__}")

print("\n✓ NumPy imported")
print("✓ Pandas imported")
print("✓ PyTorch imported")
print("✓ PyTorch Dataset/DataLoader imported")
print("✓ Evaluation metrics imported")

print("\n" + "=" * 100)
print("✓ ALL REQUIRED LIBRARIES IMPORTED")
print("=" * 100)

PHASE 03.04 — BASELINE CROSS-MODAL TRAINING

LIBRARY STATUS
----------------------------------------------------------------------------------------------------
NumPy       : 2.5.2
Pandas      : 2.3.3
PyTorch     : 2.13.0+cpu

✓ NumPy imported
✓ Pandas imported
✓ PyTorch imported
✓ PyTorch Dataset/DataLoader imported
✓ Evaluation metrics imported

✓ ALL REQUIRED LIBRARIES IMPORTED


In [11]:
# ==================================================================================================
# CELL 2 — CONFIGURE REPRODUCIBILITY
# ==================================================================================================

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("=" * 100)
print("REPRODUCIBILITY CONFIGURATION")
print("=" * 100)

print(f"\nRandom seed : {RANDOM_SEED}")
print(f"CUDA        : {torch.cuda.is_available()}")

print("\n✓ Python random seed configured")
print("✓ NumPy random seed configured")
print("✓ PyTorch random seed configured")

if torch.cuda.is_available():
    print("✓ PyTorch CUDA seed configured")

print("\n" + "=" * 100)
print("✓ REPRODUCIBILITY CONFIGURATION COMPLETED")
print("=" * 100)

REPRODUCIBILITY CONFIGURATION

Random seed : 42
CUDA        : False

✓ Python random seed configured
✓ NumPy random seed configured
✓ PyTorch random seed configured

✓ REPRODUCIBILITY CONFIGURATION COMPLETED


In [12]:
# ==================================================================================================
# CELL 3 — DEFINE PROJECT ROOT AND BASELINE ARTIFACT PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    r"C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening"
)

PAIR_DATA_DIR = (
    PROJECT_ROOT /
    "06_cross_modal_learning" /
    "02_cross_modal_pairing" /
    "pairs" /
    "representation_aware_split"
)

TRAIN_PAIR_PATH = (
    PAIR_DATA_DIR /
    "02_08_train_representation_disjoint_pairs.npz"
)

VALIDATION_PAIR_PATH = (
    PAIR_DATA_DIR /
    "02_08_validation_representation_disjoint_pairs.npz"
)

BASELINE_OUTPUT_DIR = (
    PROJECT_ROOT /
    "06_cross_modal_learning" /
    "03_cross_modal_learning" /
    "03_04_baseline_training"
)

BASELINE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 100)
print("PROJECT AND ARTIFACT PATH CONFIGURATION")
print("=" * 100)

print("\nPROJECT ROOT")
print("-" * 100)
print(PROJECT_ROOT)

print("\nTRAINING DATASET")
print("-" * 100)
print(TRAIN_PAIR_PATH)

print("\nVALIDATION DATASET")
print("-" * 100)
print(VALIDATION_PAIR_PATH)

print("\nBASELINE OUTPUT DIRECTORY")
print("-" * 100)
print(BASELINE_OUTPUT_DIR)

print("\n" + "=" * 100)

assert PROJECT_ROOT.exists(), (
    "PROJECT_ROOT does not exist."
)

assert TRAIN_PAIR_PATH.exists(), (
    "Training pair dataset not found."
)

assert VALIDATION_PAIR_PATH.exists(), (
    "Validation pair dataset not found."
)

print("✓ PROJECT ROOT VERIFIED")
print("✓ TRAINING DATASET FOUND")
print("✓ VALIDATION DATASET FOUND")
print("✓ BASELINE OUTPUT DIRECTORY READY")

print("=" * 100)

PROJECT AND ARTIFACT PATH CONFIGURATION

PROJECT ROOT
----------------------------------------------------------------------------------------------------
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

TRAINING DATASET
----------------------------------------------------------------------------------------------------
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_train_representation_disjoint_pairs.npz

VALIDATION DATASET
----------------------------------------------------------------------------------------------------
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_validation_representation_disjoint_pairs.npz

BASELINE OUTPUT DIRECTORY
----------------------------------------------------------------------------------------------------
C:\Users\s

In [13]:
# ==================================================================================================
# CELL 4 — LOAD TRAINING AND VALIDATION PAIR DATASETS
# ==================================================================================================

print("=" * 100)
print("LOADING REPRESENTATION-AWARE CROSS-MODAL DATASETS")
print("=" * 100)

train_data = np.load(TRAIN_PAIR_PATH)
validation_data = np.load(VALIDATION_PAIR_PATH)

# --------------------------------------------------------------------------------
# TRAINING DATA
# --------------------------------------------------------------------------------

X_source_train = train_data["source_embeddings"].astype(np.float32)
X_target_train = train_data["target_embeddings"].astype(np.float32)
y_train = train_data["pair_labels"].astype(np.float32)

# --------------------------------------------------------------------------------
# VALIDATION DATA
# --------------------------------------------------------------------------------

X_source_val = validation_data["source_embeddings"].astype(np.float32)
X_target_val = validation_data["target_embeddings"].astype(np.float32)
y_val = validation_data["pair_labels"].astype(np.float32)


print("\nTRAINING DATASET")
print("-" * 100)

print(f"Source embeddings : {X_source_train.shape}")
print(f"Target embeddings : {X_target_train.shape}")
print(f"Pair labels       : {y_train.shape}")

print("\nVALIDATION DATASET")
print("-" * 100)

print(f"Source embeddings : {X_source_val.shape}")
print(f"Target embeddings : {X_target_val.shape}")
print(f"Pair labels       : {y_val.shape}")


print("\n" + "=" * 100)
print("✓ TRAINING DATA LOADED")
print("✓ VALIDATION DATA LOADED")
print("=" * 100)

LOADING REPRESENTATION-AWARE CROSS-MODAL DATASETS

TRAINING DATASET
----------------------------------------------------------------------------------------------------
Source embeddings : (1776, 32)
Target embeddings : (1776, 32)
Pair labels       : (1776,)

VALIDATION DATASET
----------------------------------------------------------------------------------------------------
Source embeddings : (448, 32)
Target embeddings : (448, 32)
Pair labels       : (448,)

✓ TRAINING DATA LOADED
✓ VALIDATION DATA LOADED


In [14]:
# ==================================================================================================
# CELL 5 — PRE-TRAINING DATA VALIDATION
# ==================================================================================================

print("=" * 100)
print("PRE-TRAINING DATA VALIDATION")
print("=" * 100)

# --------------------------------------------------------------------------------
# SHAPE CHECKS
# --------------------------------------------------------------------------------

assert X_source_train.shape == X_target_train.shape, (
    "Training source and target shapes do not match."
)

assert X_source_val.shape == X_target_val.shape, (
    "Validation source and target shapes do not match."
)

assert X_source_train.shape[1] == 32, (
    "Training source embeddings are not 32-dimensional."
)

assert X_target_train.shape[1] == 32, (
    "Training target embeddings are not 32-dimensional."
)

assert X_source_val.shape[1] == 32, (
    "Validation source embeddings are not 32-dimensional."
)

assert X_target_val.shape[1] == 32, (
    "Validation target embeddings are not 32-dimensional."
)


# --------------------------------------------------------------------------------
# ROW / LABEL CORRESPONDENCE
# --------------------------------------------------------------------------------

assert len(X_source_train) == len(y_train), (
    "Training source embeddings and labels have different row counts."
)

assert len(X_target_train) == len(y_train), (
    "Training target embeddings and labels have different row counts."
)

assert len(X_source_val) == len(y_val), (
    "Validation source embeddings and labels have different row counts."
)

assert len(X_target_val) == len(y_val), (
    "Validation target embeddings and labels have different row counts."
)


# --------------------------------------------------------------------------------
# LABEL VALIDATION
# --------------------------------------------------------------------------------

assert set(np.unique(y_train)).issubset({0.0, 1.0}), (
    "Training labels contain values other than 0 and 1."
)

assert set(np.unique(y_val)).issubset({0.0, 1.0}), (
    "Validation labels contain values other than 0 and 1."
)


# --------------------------------------------------------------------------------
# NUMERICAL VALIDATION
# --------------------------------------------------------------------------------

assert np.isfinite(X_source_train).all(), (
    "Training source embeddings contain NaN or infinite values."
)

assert np.isfinite(X_target_train).all(), (
    "Training target embeddings contain NaN or infinite values."
)

assert np.isfinite(X_source_val).all(), (
    "Validation source embeddings contain NaN or infinite values."
)

assert np.isfinite(X_target_val).all(), (
    "Validation target embeddings contain NaN or infinite values."
)


# --------------------------------------------------------------------------------
# CLASS DISTRIBUTION
# --------------------------------------------------------------------------------

train_positive = int(np.sum(y_train == 1))
train_negative = int(np.sum(y_train == 0))

val_positive = int(np.sum(y_val == 1))
val_negative = int(np.sum(y_val == 0))


print("\nDATASET COUNTS")
print("-" * 100)

print(f"Training pairs        : {len(y_train)}")
print(f"Training positive     : {train_positive}")
print(f"Training negative     : {train_negative}")

print(f"\nValidation pairs      : {len(y_val)}")
print(f"Validation positive   : {val_positive}")
print(f"Validation negative   : {val_negative}")


print("\nVALIDATION RESULTS")
print("-" * 100)

print("✓ Source / target dimensions verified")
print("✓ 32D embedding structure verified")
print("✓ Source / target / label correspondence verified")
print("✓ Binary labels verified")
print("✓ NaN / infinite values absent")
print("✓ Training / validation dataset sizes verified")


print("\n" + "=" * 100)
print("✓ PRE-TRAINING DATA VALIDATION PASSED")
print("✓ DATA IS READY FOR BASELINE TRAINING")
print("=" * 100)

PRE-TRAINING DATA VALIDATION

DATASET COUNTS
----------------------------------------------------------------------------------------------------
Training pairs        : 1776
Training positive     : 888
Training negative     : 888

Validation pairs      : 448
Validation positive   : 224
Validation negative   : 224

VALIDATION RESULTS
----------------------------------------------------------------------------------------------------
✓ Source / target dimensions verified
✓ 32D embedding structure verified
✓ Source / target / label correspondence verified
✓ Binary labels verified
✓ NaN / infinite values absent
✓ Training / validation dataset sizes verified

✓ PRE-TRAINING DATA VALIDATION PASSED
✓ DATA IS READY FOR BASELINE TRAINING


In [15]:
# ============================================================
# CELL 6 — RECREATE BASELINE CROSS-MODAL MODEL
# ============================================================

# Select computation device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


class BaselineCrossModalModel(nn.Module):
    """
    Baseline cross-modal relationship classifier.

    Input:
        source_embedding : (batch_size, 32)
        target_embedding : (batch_size, 32)

    Feature construction:
        [S, T, |S-T|, S*T] -> 128 dimensions

    Classifier:
        128 -> 128 -> 64 -> 1
    """

    def __init__(self):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, source, target):

        difference = torch.abs(source - target)

        interaction = source * target

        combined = torch.cat(
            [source, target, difference, interaction],
            dim=1
        )

        output = self.classifier(combined)

        return output.squeeze(1)


# Create model
model = BaselineCrossModalModel().to(device)

print("=" * 100)
print("BASELINE MODEL CREATED")
print("=" * 100)

print(model)

Using device: cpu
BASELINE MODEL CREATED
BaselineCrossModalModel(
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [21]:
# ============================================================
# CELL 6A — LOAD BASELINE PAIR DATA
# SELF-CONTAINED VERSION
# ============================================================

from pathlib import Path
import numpy as np


# ------------------------------------------------------------
# STEP 1 — Locate the project root
# ------------------------------------------------------------

current_dir = Path.cwd()

print("=" * 100)
print("CURRENT NOTEBOOK DIRECTORY")
print("=" * 100)
print(current_dir)


# ------------------------------------------------------------
# STEP 2 — Search upward for the project folder
# ------------------------------------------------------------

project_root = None

for path in [current_dir] + list(current_dir.parents):

    candidate = (
        path
        / "06_cross_modal_learning"
        / "02_cross_modal_pairing"
        / "pairs"
        / "representation_aware_split"
    )

    if candidate.exists():
        project_root = path
        break


# ------------------------------------------------------------
# STEP 3 — Stop if project root cannot be found
# ------------------------------------------------------------

if project_root is None:
    raise FileNotFoundError(
        "Could not automatically locate the ASD Multimodal project root.\n"
        "The folder '06_cross_modal_learning/02_cross_modal_pairing/"
        "pairs/representation_aware_split' was not found."
    )


print("\nProject root found:")
print(project_root)


# ------------------------------------------------------------
# STEP 4 — Define exact artifact paths
# ------------------------------------------------------------

pair_dir = (
    project_root
    / "06_cross_modal_learning"
    / "02_cross_modal_pairing"
    / "pairs"
    / "representation_aware_split"
)

train_npz_path = (
    pair_dir
    / "02_08_train_representation_disjoint_pairs.npz"
)

val_npz_path = (
    pair_dir
    / "02_08_validation_representation_disjoint_pairs.npz"
)


# ------------------------------------------------------------
# STEP 5 — Verify artifact files exist
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("ARTIFACT CHECK")
print("=" * 100)

print("Training file:")
print(train_npz_path)
print("Exists:", train_npz_path.exists())

print("\nValidation file:")
print(val_npz_path)
print("Exists:", val_npz_path.exists())


if not train_npz_path.exists():
    raise FileNotFoundError(
        f"Training pair file not found:\n{train_npz_path}"
    )

if not val_npz_path.exists():
    raise FileNotFoundError(
        f"Validation pair file not found:\n{val_npz_path}"
    )


# ------------------------------------------------------------
# STEP 6 — Load training data
# ------------------------------------------------------------

train_data = np.load(train_npz_path)

train_source = train_data["source_embeddings"]
train_target = train_data["target_embeddings"]
train_labels = train_data["pair_labels"]


# ------------------------------------------------------------
# STEP 7 — Load validation data
# ------------------------------------------------------------

val_data = np.load(val_npz_path)

val_source = val_data["source_embeddings"]
val_target = val_data["target_embeddings"]
val_labels = val_data["pair_labels"]


# ------------------------------------------------------------
# STEP 8 — Validate loaded arrays
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("BASELINE TRAINING DATA LOADED")
print("=" * 100)

print("\nTRAINING")
print("-" * 100)
print("Source embeddings :", train_source.shape)
print("Target embeddings :", train_target.shape)
print("Pair labels       :", train_labels.shape)

print("\nVALIDATION")
print("-" * 100)
print("Source embeddings :", val_source.shape)
print("Target embeddings :", val_target.shape)
print("Pair labels       :", val_labels.shape)

print("\nCLASS DISTRIBUTION")
print("-" * 100)

print(
    "Training positive :",
    int(np.sum(train_labels == 1))
)

print(
    "Training negative :",
    int(np.sum(train_labels == 0))
)

print(
    "Validation positive :",
    int(np.sum(val_labels == 1))
)

print(
    "Validation negative :",
    int(np.sum(val_labels == 0))
)


print("\n" + "=" * 100)
print("✓ BASELINE PAIR DATA LOADING COMPLETE")
print("=" * 100)

CURRENT NOTEBOOK DIRECTORY
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_training\notebooks

Project root found:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

ARTIFACT CHECK
Training file:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_train_representation_disjoint_pairs.npz
Exists: True

Validation file:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_validation_representation_disjoint_pairs.npz
Exists: True

BASELINE TRAINING DATA LOADED

TRAINING
----------------------------------------------------------------------------------------------------
Source embeddings : (1776, 32)
Target embeddings : (1776, 32)
Pair labels       : (1776,)

VALIDATION
--------------------------------------

In [22]:
# ============================================================
# CELL 7 — BASELINE FORWARD PASS TEST
# ============================================================

model.eval()

with torch.no_grad():

    test_source = torch.tensor(
        train_source[:32],
        dtype=torch.float32
    ).to(device)

    test_target = torch.tensor(
        train_target[:32],
        dtype=torch.float32
    ).to(device)

    test_output = model(
        test_source,
        test_target
    )

print("=" * 100)
print("FORWARD PASS TEST")
print("=" * 100)

print("Source batch shape :", tuple(test_source.shape))
print("Target batch shape :", tuple(test_target.shape))
print("Output shape       :", tuple(test_output.shape))

print("\nFirst 5 logits:")
print(test_output[:5])

FORWARD PASS TEST
Source batch shape : (32, 32)
Target batch shape : (32, 32)
Output shape       : (32,)

First 5 logits:
tensor([0.0996, 0.1113, 0.1294, 0.1088, 0.0885])


In [23]:
# ============================================================
# CELL 8 — VERIFY BCE WITH LOGITS LOSS
# ============================================================

# Put the model in evaluation mode for this test
model.eval()

# Create the loss function
criterion = nn.BCEWithLogitsLoss()

# Take the first 32 training labels
test_labels = torch.tensor(
    train_labels[:32],
    dtype=torch.float32
).to(device)

# Calculate loss
test_loss = criterion(
    test_output,
    test_labels
)

print("=" * 100)
print("BASELINE LOSS VERIFICATION")
print("=" * 100)

print("Logits shape :", tuple(test_output.shape))
print("Labels shape :", tuple(test_labels.shape))

print("Loss value   :", test_loss.item())

print("\nLoss is finite:",
      bool(torch.isfinite(test_loss)))

BASELINE LOSS VERIFICATION
Logits shape : (32,)
Labels shape : (32,)
Loss value   : 0.6417804956436157

Loss is finite: True


In [24]:
# ============================================================
# CELL 9 — CONFIGURE OPTIMIZER
# ============================================================

learning_rate = 1e-3

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

print("=" * 100)
print("OPTIMIZER CONFIGURATION")
print("=" * 100)

print("Optimizer     : Adam")
print("Learning rate :", learning_rate)

print("\nOptimizer successfully attached to model parameters.")

OPTIMIZER CONFIGURATION
Optimizer     : Adam
Learning rate : 0.001

Optimizer successfully attached to model parameters.


In [25]:
# ============================================================
# CELL 10 — CREATE DATASETS AND DATALOADERS
# ============================================================

class CrossModalPairDataset(Dataset):

    def __init__(
        self,
        source_embeddings,
        target_embeddings,
        labels
    ):
        self.source = torch.tensor(
            source_embeddings,
            dtype=torch.float32
        )

        self.target = torch.tensor(
            target_embeddings,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):

        return (
            self.source[index],
            self.target[index],
            self.labels[index]
        )


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = CrossModalPairDataset(
    train_source,
    train_target,
    train_labels
)

val_dataset = CrossModalPairDataset(
    val_source,
    val_target,
    val_labels
)


# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

batch_size = 32


# ------------------------------------------------------------
# Create DataLoaders
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)


print("=" * 100)
print("DATALOADERS CREATED")
print("=" * 100)

print("Training samples   :", len(train_dataset))
print("Validation samples :", len(val_dataset))

print("Training batches   :", len(train_loader))
print("Validation batches :", len(val_loader))

print("Batch size         :", batch_size)

DATALOADERS CREATED
Training samples   : 1776
Validation samples : 448
Training batches   : 56
Validation batches : 14
Batch size         : 32


In [26]:
# ============================================================
# CELL 11 — GET ONE TRAINING BATCH
# ============================================================

model.train()

source_batch, target_batch, label_batch = next(
    iter(train_loader)
)

source_batch = source_batch.to(device)
target_batch = target_batch.to(device)
label_batch = label_batch.to(device)


print("=" * 100)
print("SINGLE TRAINING BATCH")
print("=" * 100)

print("Source batch shape :", tuple(source_batch.shape))
print("Target batch shape :", tuple(target_batch.shape))
print("Label batch shape  :", tuple(label_batch.shape))

print("\nPositive labels in batch:",
      int((label_batch == 1).sum().item()))

print("Negative labels in batch:",
      int((label_batch == 0).sum().item()))

SINGLE TRAINING BATCH
Source batch shape : (32, 32)
Target batch shape : (32, 32)
Label batch shape  : (32,)

Positive labels in batch: 12
Negative labels in batch: 20


In [27]:
# ============================================================
# CELL 12 — FORWARD PASS AND TRAINING LOSS
# ============================================================

# Generate relationship logits
batch_logits = model(
    source_batch,
    target_batch
)

# Calculate binary classification loss
batch_loss = criterion(
    batch_logits,
    label_batch
)


print("=" * 100)
print("TRAINING BATCH FORWARD PASS")
print("=" * 100)

print("Logits shape :", tuple(batch_logits.shape))
print("Labels shape :", tuple(label_batch.shape))

print("Batch loss   :", batch_loss.item())

print("Loss finite  :",
      bool(torch.isfinite(batch_loss)))

TRAINING BATCH FORWARD PASS
Logits shape : (32,)
Labels shape : (32,)
Batch loss   : 0.7421303391456604
Loss finite  : True


In [28]:
# ============================================================
# CELL 13 — BACKPROPAGATION
# ============================================================

# Clear any existing gradients
optimizer.zero_grad()

# Calculate gradients
batch_loss.backward()


print("=" * 100)
print("BACKPROPAGATION CHECK")
print("=" * 100)

print("Gradients calculated successfully.")

# Check whether gradients are finite
all_gradients_finite = True
gradient_count = 0

for name, parameter in model.named_parameters():

    if parameter.grad is not None:

        gradient_count += 1

        if not torch.all(torch.isfinite(parameter.grad)):
            all_gradients_finite = False

        print(
            f"{name:30s} "
            f"gradient shape = {tuple(parameter.grad.shape)}"
        )


print("\nParameters with gradients :", gradient_count)
print("All gradients finite      :", all_gradients_finite)

BACKPROPAGATION CHECK
Gradients calculated successfully.
classifier.0.weight            gradient shape = (128, 128)
classifier.0.bias              gradient shape = (128,)
classifier.2.weight            gradient shape = (64, 128)
classifier.2.bias              gradient shape = (64,)
classifier.4.weight            gradient shape = (1, 64)
classifier.4.bias              gradient shape = (1,)

Parameters with gradients : 6
All gradients finite      : True


In [29]:
# ============================================================
# CELL 14 — FIRST OPTIMIZER UPDATE
# ============================================================

# Save a copy of one parameter before the update
parameter_before = (
    model.classifier[0].weight
    .detach()
    .clone()
)

# Perform one Adam update
optimizer.step()

# Save the same parameter after the update
parameter_after = (
    model.classifier[0].weight
    .detach()
    .clone()
)


# Calculate how much the parameter changed
parameter_change = torch.abs(
    parameter_after - parameter_before
).sum().item()


print("=" * 100)
print("FIRST OPTIMIZER UPDATE")
print("=" * 100)

print("Parameter change:",
      parameter_change)

print(
    "Weights changed:",
    parameter_change > 0
)

FIRST OPTIMIZER UPDATE
Parameter change: 14.948832511901855
Weights changed: True


In [30]:
# ============================================================
# CELL 15 — VERIFY UPDATED MODEL
# ============================================================

model.eval()

with torch.no_grad():

    updated_logits = model(
        source_batch,
        target_batch
    )

    updated_loss = criterion(
        updated_logits,
        label_batch
    )


print("=" * 100)
print("POST-UPDATE VERIFICATION")
print("=" * 100)

print("Updated logits shape:",
      tuple(updated_logits.shape))

print("Updated loss:",
      updated_loss.item())

print(
    "Updated loss finite:",
    bool(torch.isfinite(updated_loss))
)

print(
    "Model parameters remain finite:",
    all(
        torch.all(torch.isfinite(parameter))
        for parameter in model.parameters()
    )
)

POST-UPDATE VERIFICATION
Updated logits shape: (32,)
Updated loss: 0.7055182456970215
Updated loss finite: True
Model parameters remain finite: True


In [31]:
# ============================================================
# CELL 16 — RESET MODEL FOR OFFICIAL BASELINE TRAINING
# ============================================================

# Recreate the baseline model from fresh random initialization
model = BaselineCrossModalModel().to(device)

# Recreate the optimizer so it has no history from the
# single-step verification experiment
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)


print("=" * 100)
print("BASELINE MODEL RESET")
print("=" * 100)

print("Model recreated        : True")
print("Optimizer recreated    : True")
print("Learning rate          :", learning_rate)
print("Previous test update  : Discarded")
print("Official training      : Starts from fresh initialization")

BASELINE MODEL RESET
Model recreated        : True
Optimizer recreated    : True
Learning rate          : 0.001
Previous test update  : Discarded
Official training      : Starts from fresh initialization


In [32]:
# ============================================================
# CELL 17 — INITIALIZE TRAINING HISTORY
# ============================================================

num_epochs = 50

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_balanced_accuracy": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": [],
    "val_roc_auc": []
}

print("=" * 100)
print("BASELINE TRAINING HISTORY INITIALIZED")
print("=" * 100)

print("Number of epochs:", num_epochs)

print("\nMetrics to record:")
for metric_name in history.keys():
    print(" -", metric_name)

BASELINE TRAINING HISTORY INITIALIZED
Number of epochs: 50

Metrics to record:
 - epoch
 - train_loss
 - val_loss
 - val_accuracy
 - val_balanced_accuracy
 - val_precision
 - val_recall
 - val_f1
 - val_roc_auc


In [33]:
# ============================================================
# CELL 18 — OFFICIAL BASELINE TRAINING LOOP
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("=" * 100)
print("STARTING OFFICIAL BASELINE TRAINING")
print("=" * 100)

for epoch in range(1, num_epochs + 1):

    # ========================================================
    # TRAINING PHASE
    # ========================================================

    model.train()

    running_train_loss = 0.0
    train_sample_count = 0

    for source_batch, target_batch, label_batch in train_loader:

        # Move data to selected device
        source_batch = source_batch.to(device)
        target_batch = target_batch.to(device)
        label_batch = label_batch.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(
            source_batch,
            target_batch
        )

        # Calculate loss
        loss = criterion(
            logits,
            label_batch
        )

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        # Accumulate loss
        batch_size_actual = label_batch.size(0)

        running_train_loss += (
            loss.item() * batch_size_actual
        )

        train_sample_count += batch_size_actual


    # Average training loss across all samples
    train_loss = (
        running_train_loss /
        train_sample_count
    )


    # ========================================================
    # VALIDATION PHASE
    # ========================================================

    model.eval()

    running_val_loss = 0.0
    val_sample_count = 0

    all_val_labels = []
    all_val_logits = []

    with torch.no_grad():

        for source_batch, target_batch, label_batch in val_loader:

            # Move validation data to device
            source_batch = source_batch.to(device)
            target_batch = target_batch.to(device)
            label_batch = label_batch.to(device)

            # Forward pass only
            logits = model(
                source_batch,
                target_batch
            )

            # Validation loss
            loss = criterion(
                logits,
                label_batch
            )

            # Accumulate validation loss
            batch_size_actual = label_batch.size(0)

            running_val_loss += (
                loss.item() * batch_size_actual
            )

            val_sample_count += batch_size_actual

            # Store predictions for metrics
            all_val_labels.extend(
                label_batch.cpu().numpy()
            )

            all_val_logits.extend(
                logits.cpu().numpy()
            )


    # Average validation loss
    val_loss = (
        running_val_loss /
        val_sample_count
    )


    # ========================================================
    # CONVERT LOGITS TO PROBABILITIES
    # ========================================================

    all_val_labels = np.asarray(
        all_val_labels
    ).astype(int)

    all_val_logits = np.asarray(
        all_val_logits
    )

    # Sigmoid
    all_val_probabilities = (
        1.0 /
        (1.0 + np.exp(-all_val_logits))
    )

    # Classification threshold
    all_val_predictions = (
        all_val_probabilities >= 0.5
    ).astype(int)


    # ========================================================
    # CALCULATE VALIDATION METRICS
    # ========================================================

    val_accuracy = accuracy_score(
        all_val_labels,
        all_val_predictions
    )

    val_balanced_accuracy = balanced_accuracy_score(
        all_val_labels,
        all_val_predictions
    )

    val_precision = precision_score(
        all_val_labels,
        all_val_predictions,
        zero_division=0
    )

    val_recall = recall_score(
        all_val_labels,
        all_val_predictions,
        zero_division=0
    )

    val_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        zero_division=0
    )

    val_roc_auc = roc_auc_score(
        all_val_labels,
        all_val_probabilities
    )


    # ========================================================
    # SAVE HISTORY
    # ========================================================

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)
    history["val_balanced_accuracy"].append(
        val_balanced_accuracy
    )
    history["val_precision"].append(
        val_precision
    )
    history["val_recall"].append(
        val_recall
    )
    history["val_f1"].append(
        val_f1
    )
    history["val_roc_auc"].append(
        val_roc_auc
    )


    # ========================================================
    # PRINT PROGRESS
    # ========================================================

    print(
        f"Epoch {epoch:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val F1: {val_f1:.4f} | "
        f"Val ROC-AUC: {val_roc_auc:.4f}"
    )


print("\n" + "=" * 100)
print("✓ OFFICIAL BASELINE TRAINING COMPLETE")
print("=" * 100)

STARTING OFFICIAL BASELINE TRAINING
Epoch 01/50 | Train Loss: 0.6955 | Val Loss: 0.6911 | Val Acc: 0.5625 | Val F1: 0.6157 | Val ROC-AUC: 0.5484
Epoch 02/50 | Train Loss: 0.6922 | Val Loss: 0.6913 | Val Acc: 0.5045 | Val F1: 0.6626 | Val ROC-AUC: 0.5694
Epoch 03/50 | Train Loss: 0.6877 | Val Loss: 0.6897 | Val Acc: 0.5179 | Val F1: 0.6646 | Val ROC-AUC: 0.6002
Epoch 04/50 | Train Loss: 0.6847 | Val Loss: 0.6790 | Val Acc: 0.5915 | Val F1: 0.4816 | Val ROC-AUC: 0.5828
Epoch 05/50 | Train Loss: 0.6680 | Val Loss: 0.6664 | Val Acc: 0.6183 | Val F1: 0.5938 | Val ROC-AUC: 0.6415
Epoch 06/50 | Train Loss: 0.6602 | Val Loss: 0.6684 | Val Acc: 0.5781 | Val F1: 0.4112 | Val ROC-AUC: 0.6491
Epoch 07/50 | Train Loss: 0.6479 | Val Loss: 0.6780 | Val Acc: 0.5915 | Val F1: 0.3838 | Val ROC-AUC: 0.6539
Epoch 08/50 | Train Loss: 0.6337 | Val Loss: 0.6479 | Val Acc: 0.6183 | Val F1: 0.5919 | Val ROC-AUC: 0.6620
Epoch 09/50 | Train Loss: 0.6242 | Val Loss: 0.6513 | Val Acc: 0.6094 | Val F1: 0.5383 | Val

In [34]:
# ============================================================
# CELL 19 — BEST EPOCH ANALYSIS
# ============================================================

import numpy as np


print("=" * 100)
print("BASELINE TRAINING HISTORY ANALYSIS")
print("=" * 100)


# ------------------------------------------------------------
# Convert history lists to NumPy arrays
# ------------------------------------------------------------

epochs = np.asarray(history["epoch"])

train_losses = np.asarray(history["train_loss"])
val_losses = np.asarray(history["val_loss"])

val_accuracy = np.asarray(history["val_accuracy"])
val_balanced_accuracy = np.asarray(
    history["val_balanced_accuracy"]
)

val_precision = np.asarray(history["val_precision"])
val_recall = np.asarray(history["val_recall"])
val_f1 = np.asarray(history["val_f1"])
val_roc_auc = np.asarray(history["val_roc_auc"])


# ------------------------------------------------------------
# Find best epoch for each validation metric
# ------------------------------------------------------------

best_loss_index = np.argmin(val_losses)
best_accuracy_index = np.argmax(val_accuracy)
best_balanced_accuracy_index = np.argmax(
    val_balanced_accuracy
)
best_precision_index = np.argmax(val_precision)
best_recall_index = np.argmax(val_recall)
best_f1_index = np.argmax(val_f1)
best_roc_auc_index = np.argmax(val_roc_auc)


# ------------------------------------------------------------
# Print best results
# ------------------------------------------------------------

print("\nBEST VALIDATION RESULTS")
print("-" * 100)

print(
    f"Best Validation Loss        : "
    f"Epoch {epochs[best_loss_index]:02d} | "
    f"{val_losses[best_loss_index]:.4f}"
)

print(
    f"Best Validation Accuracy    : "
    f"Epoch {epochs[best_accuracy_index]:02d} | "
    f"{val_accuracy[best_accuracy_index]:.4f}"
)

print(
    f"Best Balanced Accuracy      : "
    f"Epoch {epochs[best_balanced_accuracy_index]:02d} | "
    f"{val_balanced_accuracy[best_balanced_accuracy_index]:.4f}"
)

print(
    f"Best Precision              : "
    f"Epoch {epochs[best_precision_index]:02d} | "
    f"{val_precision[best_precision_index]:.4f}"
)

print(
    f"Best Recall                 : "
    f"Epoch {epochs[best_recall_index]:02d} | "
    f"{val_recall[best_recall_index]:.4f}"
)

print(
    f"Best F1                     : "
    f"Epoch {epochs[best_f1_index]:02d} | "
    f"{val_f1[best_f1_index]:.4f}"
)

print(
    f"Best ROC-AUC                : "
    f"Epoch {epochs[best_roc_auc_index]:02d} | "
    f"{val_roc_auc[best_roc_auc_index]:.4f}"
)


# ------------------------------------------------------------
# Primary baseline selection criterion
# ------------------------------------------------------------

best_epoch = int(
    epochs[best_roc_auc_index]
)

best_roc_auc = float(
    val_roc_auc[best_roc_auc_index]
)


print("\n" + "=" * 100)
print("PRIMARY BASELINE MODEL SELECTION")
print("=" * 100)

print("Selection criterion : Validation ROC-AUC")
print(f"Selected epoch     : {best_epoch}")
print(f"Best validation ROC-AUC : {best_roc_auc:.4f}")


# ------------------------------------------------------------
# Compare selected epoch with final epoch
# ------------------------------------------------------------

final_index = -1

print("\n" + "=" * 100)
print("SELECTED EPOCH VS FINAL EPOCH")
print("=" * 100)

print(
    f"Selected Epoch {best_epoch:02d} | "
    f"Train Loss: {train_losses[best_roc_auc_index]:.4f} | "
    f"Val Loss: {val_losses[best_roc_auc_index]:.4f} | "
    f"Val ROC-AUC: {val_roc_auc[best_roc_auc_index]:.4f}"
)

print(
    f"Final Epoch    50 | "
    f"Train Loss: {train_losses[final_index]:.4f} | "
    f"Val Loss: {val_losses[final_index]:.4f} | "
    f"Val ROC-AUC: {val_roc_auc[final_index]:.4f}"
)


print("\n" + "=" * 100)
print("✓ BEST EPOCH ANALYSIS COMPLETE")
print("=" * 100)

BASELINE TRAINING HISTORY ANALYSIS

BEST VALIDATION RESULTS
----------------------------------------------------------------------------------------------------
Best Validation Loss        : Epoch 08 | 0.6479
Best Validation Accuracy    : Epoch 35 | 0.6473
Best Balanced Accuracy      : Epoch 35 | 0.6473
Best Precision              : Epoch 07 | 0.7808
Best Recall                 : Epoch 02 | 0.9732
Best F1                     : Epoch 03 | 0.6646
Best ROC-AUC                : Epoch 16 | 0.6743

PRIMARY BASELINE MODEL SELECTION
Selection criterion : Validation ROC-AUC
Selected epoch     : 16
Best validation ROC-AUC : 0.6743

SELECTED EPOCH VS FINAL EPOCH
Selected Epoch 16 | Train Loss: 0.5759 | Val Loss: 0.6494 | Val ROC-AUC: 0.6743
Final Epoch    50 | Train Loss: 0.3483 | Val Loss: 1.0545 | Val ROC-AUC: 0.6513

✓ BEST EPOCH ANALYSIS COMPLETE


In [35]:
# ============================================================
# CELL 20 — RECREATE SELECTED EPOCH-16 BASELINE MODEL
# ============================================================

import copy
import random
import numpy as np
import torch


print("=" * 100)
print("RECREATING SELECTED BASELINE CHECKPOINT")
print("=" * 100)

print("Target epoch       : 16")
print("Selection criterion: Validation ROC-AUC")
print("Target ROC-AUC     : 0.6743")


# ------------------------------------------------------------
# Reset random seeds
# ------------------------------------------------------------

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


# ------------------------------------------------------------
# Recreate fresh model
# ------------------------------------------------------------

best_model = BaselineCrossModalModel().to(device)

best_optimizer = torch.optim.Adam(
    best_model.parameters(),
    lr=learning_rate
)


# ------------------------------------------------------------
# Recreate training loop up to Epoch 16
# ------------------------------------------------------------

for epoch in range(1, 17):

    best_model.train()

    for source_batch, target_batch, label_batch in train_loader:

        source_batch = source_batch.to(device)
        target_batch = target_batch.to(device)
        label_batch = label_batch.to(device)

        best_optimizer.zero_grad()

        logits = best_model(
            source_batch,
            target_batch
        )

        loss = criterion(
            logits,
            label_batch
        )

        loss.backward()

        best_optimizer.step()


# ------------------------------------------------------------
# Evaluate recreated model
# ------------------------------------------------------------

best_model.eval()

recreated_val_logits = []
recreated_val_labels = []

with torch.no_grad():

    for source_batch, target_batch, label_batch in val_loader:

        source_batch = source_batch.to(device)
        target_batch = target_batch.to(device)

        logits = best_model(
            source_batch,
            target_batch
        )

        recreated_val_logits.extend(
            logits.cpu().numpy()
        )

        recreated_val_labels.extend(
            label_batch.numpy()
        )


recreated_val_logits = np.asarray(
    recreated_val_logits
)

recreated_val_labels = np.asarray(
    recreated_val_labels
).astype(int)


recreated_val_probabilities = (
    1.0 /
    (1.0 + np.exp(-recreated_val_logits))
)


recreated_val_predictions = (
    recreated_val_probabilities >= 0.5
).astype(int)


recreated_roc_auc = roc_auc_score(
    recreated_val_labels,
    recreated_val_probabilities
)


recreated_accuracy = accuracy_score(
    recreated_val_labels,
    recreated_val_predictions
)


recreated_balanced_accuracy = balanced_accuracy_score(
    recreated_val_labels,
    recreated_val_predictions
)


recreated_precision = precision_score(
    recreated_val_labels,
    recreated_val_predictions,
    zero_division=0
)


recreated_recall = recall_score(
    recreated_val_labels,
    recreated_val_predictions,
    zero_division=0
)


recreated_f1 = f1_score(
    recreated_val_labels,
    recreated_val_predictions,
    zero_division=0
)


print("\n" + "=" * 100)
print("RECREATED EPOCH-16 VALIDATION RESULTS")
print("=" * 100)

print(f"ROC-AUC            : {recreated_roc_auc:.4f}")
print(f"Accuracy           : {recreated_accuracy:.4f}")
print(f"Balanced Accuracy  : {recreated_balanced_accuracy:.4f}")
print(f"Precision          : {recreated_precision:.4f}")
print(f"Recall             : {recreated_recall:.4f}")
print(f"F1                 : {recreated_f1:.4f}")

RECREATING SELECTED BASELINE CHECKPOINT
Target epoch       : 16
Selection criterion: Validation ROC-AUC
Target ROC-AUC     : 0.6743

RECREATED EPOCH-16 VALIDATION RESULTS
ROC-AUC            : 0.6531
Accuracy           : 0.6004
Balanced Accuracy  : 0.6004
Precision          : 0.5957
Recall             : 0.6250
F1                 : 0.6100
